# Mission 02: Multi-layered Prompt & Cache - 해답 노트북

이 노트북은 다섯 번째 미션의 완성된 솔루션 코드와 설명입니다.

In [3]:
# 1. 필요한 라이브러리 및 환경 로드
import sys
import os
from dotenv import load_dotenv

while not os.path.exists("app") and os.getcwd() != "/":
    os.chdir("..")
# 루트 폴더 기준의 경로 등록
sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("."))  # sys.path.append("app") 대신 "." 등록이 파이썬 패키지 경로 탐색에 안전합니다.
load_dotenv(override=True)

from app.utils.llm import get_llm
from langchain_core.messages import SystemMessage, HumanMessage

### [미션 1] 계층형 PromptManager 구현하기

아래 코드는 L1 ~ L4의 계층별 프롬프트 블록을 합쳐 시스템 프롬프트를 렌더링하고 동적 데이터를 치환하는 PromptManager 솔루션 코드입니다.

In [7]:
# prompts 폴더 하위의 prompt_manager 모듈로부터 PromptManager 클래스 로드
import sys
import os

# 현재 미션 폴더의 경로를 패키지 검색 경로에 추가합니다.
mission_dir = os.path.abspath("notebooks/missions/mission_02_Prompt_and_Caching")
if mission_dir not in sys.path:
    sys.path.append(mission_dir)

from prompts.prompt_manager import PromptManager

# 명시적으로 프롬프트 파일이 위치한 경로를 주입합니다.
pm = PromptManager(prompt_dir=os.path.join(mission_dir, "prompts"))
print("✅ 파일 기반 PromptManager 로드 및 객체 초기화 성공!")


✅ 파일 기반 PromptManager 로드 및 객체 초기화 성공!


### [미션 2] 프롬프트 렌더링 및 캐시 지정 시뮬레이션

대용량 더미 문서 데이터를 L4 영역에 설정하고, `PromptManager`를 통해 시스템 프롬프트를 구성해 봅니다.

In [9]:
# 대규모 정적 API 레퍼런스 문서 모사 (캐싱 대상)
large_api_docs = """=== API Reference Manual ===\n""" + "\n".join(
    [f"Function_ID_{i}: Perform operation {i}. Parameters: arg{i}. Returns result." for i in range(500)]
)

pm.set_reference_context(large_api_docs)

state = {
    "user_permission": "READ_WRITE_EXECUTE",
    "active_project": "harness_agent_lab"
}

system_prompt = pm.build_system_prompt(state)

print(f"생성된 시스템 프롬프트 크기: {len(system_prompt)} 글자")
print("\n", system_prompt)

생성된 시스템 프롬프트 크기: 40958 글자

 === ROLE (L1) ===
# 🤖 [SYSTEM DIRECTIVE: Harness Agent Spec]

You are the Harness Agent, a powerful agentic AI coding assistant designed to pair program with users to solve software engineering tasks. You operate with access to local files and system execution tools.

## 1. Core Identity & Tone Guidelines
- Engage warmly yet honestly with the user. Be direct and concise. Avoid ungrounded flattery or sycophancy.
- Respect the user's boundaries. Focus on helping them achieve autonomy and independence.
- Maintain a professional, grounded, and safety-oriented stance in all interactions.

## 2. Software Engineering Best Practices
- **Insecure Code Prevention**: Always prioritize security. Inspect all parameters for command injection, SQL injection, XSS, and other vulnerabilities before execution. Proactively fix insecure patterns.
- **Do What is Requested**: Match the scope of your changes exactly to what was asked. Avoid introducing premature abstractions or spe

### [미션 3] 에이전트 캐싱 지연시간 최적화 검증

캐싱이 걸렸을 때의 Latency 단축 효과를 확인하기 위해 시뮬레이션 테스트를 수행합니다.

In [10]:
import time
llm = get_llm(model_name="gemini-3.5-flash", temperature=0.0)

print("🔄 1회차 호출 (Cold Start - 캐시 생성) 시작...")
t1 = time.time()
res1 = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="Function_ID_256번 API의 매개변수와 반환 스펙이 무엇인지 설명해줘.")
])
cold_latency = time.time() - t1
print(f"✅ 1회차 호출 완료! (소요 시간: {cold_latency:.2f}초)")
print("답변:", res1.content)

print("\n" + "-"*50 + "\n")

print("🔄 2회차 호출 (Warm Start - 캐시 히트) 시작...")
t2 = time.time()
res2 = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content="Function_ID_128번 API의 매개변수와 반환 스펙은 뭐야?")
])
warm_latency = time.time() - t2
print(f"✅ 2회차 호출 완료! (소요 시간: {warm_latency:.2f}초)")
print("답변:", res2.content)

speedup = cold_latency / warm_latency if warm_latency > 0 else 1.0
print(f"\n🚀 캐싱 적용으로 속도가 약 {speedup:.1f}배 개선되었습니다.")

🔄 1회차 호출 (Cold Start - 캐시 생성) 시작...
✅ 1회차 호출 완료! (소요 시간: 3.35초)
답변: [{'type': 'text', 'text': '제공해주신 API 레퍼런스 매뉴얼에 따른 **Function_ID_256** API의 매개변수와 반환 스펙은 다음과 같습니다.\n\n* **기능**: Perform operation 256 (256번 작업 수행)\n* **매개변수 (Parameters)**: `arg256`\n* **반환값 (Returns)**: `result` (결과)', 'thought_signature': 'AY89a18CxSqK6n/keWkPiO5Tw8lJPrY7ag+BlNOQkmSRHcNhQESzGLluZQh0ZrwaBgki+uyPnmmUh4T/ki0p7pUSxvOIJn3fm7Dh6CbWvOQpmdb3tUJD8GcVCbW7ypvfi8DNseieW6Y0E8LG6aUfTixsVkXiyzLEPVABRAndc7KKIn39ohTFbQHkJF+zIx6TicMLo89YdDCMxKz9bVV9R17HO38SVIH9/wb9CFzkJO7ThQR7umDgvGJPe15dVhgOIiShj2VURZtHiI0IWTxvqCtvrF3f6TCKMXq4A2u8O1PJUKfwMj/c9q4403+Ga8yduUpPNTiJg8/7cIYvgbEJ6YuogWYI/13oy+YlEV4j8wlfVU2mar6q4c9I+F9gQd9Q/0QN3LxMi9JmaunYF2Pxv1dVyP4ek8bZFCR0kHTXVzKUZ/CuARdsyYLxBItKMgKZKSrPB8tVLjCUDZrXsIugEs1eL2Vh9PKKBemyICMzb9fSVvMLsX+JijNg95i4GHtpzJelQeoa3kIBFhe/cvj8JnYLZBtlRSS0rOHbd+SR6+GA8TzaSiv8A+SLGbV+Y0hEfQRsuKfgvOtzCZTsXrNToyg8dVtgjr2bKIuvcJHYR5bHU1jc6O1ekGkk+xw1xwJ5YCDlgd/e39mYym21stDnKCIGfCQtS7TRtgdH0fYBZrp63xFWcn2

### [보너스 미션]  미들웨어를 통한 자동 프롬프트 주입 검증

실제 프로덕션 환경(예: Harness Agent 실행 루프)에서는 개발자가 직접 프롬프트 매니저를 호출하지 않고, 미들웨어가 모델 실행 직전에 개입하여 동적으로 프롬프트를 구성해 공급합니다.

정의한 미들웨어 함수 가 모의 요청(Mock ModelRequest)과 런타임 콘텍스트(Runtime Context)를 받아 정상적으로 5계층 프롬프트를 조립하는지 확인해 봅니다.

In [24]:
import inspect
print(inspect.getsource(harness_agent_prompt_middleware.before_model))


    def before_model(self, state: StateT, runtime: Runtime[ContextT]) -> dict[str, Any] | None:
        """Logic to run before the model is called.

        Args:
            state: The current agent state.
            runtime: The runtime context.

        Returns:
            Agent state updates to apply before model call.
        """



In [22]:
# 미들웨어 가져오기 및 호출
from prompts.prompt_manager import harness_agent_prompt_middleware

mock_request = MockModelRequest()

# 🌟 request와 함께 mock_request.runtime을 두 번째 인자로 전달합니다.
middleware_prompt = harness_agent_prompt_middleware.before_model(mock_request, mock_request.runtime)

# 2. mock_request 내부에 주입된 속성 출력
print("✅ mock_request 내부 속성 목록:")
for k, v in mock_request.__dict__.items():
    # 문자열 타입의 속성이 있다면 프롬프트가 저장된 변수일 확률이 높습니다.
    if isinstance(v, str):
         print(f"  * {k} (문자열): {v[:100]}...")
    else:
         print(f"  * {k} ({type(v).__name__})")

✅ mock_request 내부 속성 목록:
  * runtime (MockRuntime)
  * tools (list)
